In [1]:
import os
import wandb # для логирования

import numpy as np
import random
from tqdm import *
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim # для оптимизаторов
from torchvision import datasets # для данных
import torchvision.transforms as transforms # для преобразований тензоров
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import joblib

import matplotlib.pyplot as plt

In [2]:
df= pd.read_csv('car_policy.csv')

df.head()

,policy_tenure,age_of_car,age_of_policyholder,population_density,make,max_torque,max_power,airbags,is_esc,is_adjustable_steering,...,engine_type_K Series Dual jet,engine_type_K10C,engine_type_i-DTEC,rear_brakes_type_Drum,transmission_type_Manual,steering_type_Manual,steering_type_Power,safe_score,car_size,age_of_car_and_policy
0,0.515874,0.05,0.644231,4990,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.025794
1,0.672619,0.02,0.375000,27003,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.013452
2,0.841110,0.02,0.384615,4076,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.016822
3,0.900277,0.11,0.432692,21622,1,113.0,88.50,2,1,1,...,0,0,0,1,0,0,0,6,10500957375,0.099030
4,0.596403,0.11,0.634615,34738,2,91.0,67.06,2,0,0,...,0,0,0,1,0,0,0,3,8777961010,0.065604


In [3]:
# Разделение на X и y
X = df.drop(columns = ['is_claim'])
y = df['is_claim']

print(X.shape)
print(y.shape)

(58592, 89)
(58592,)


In [4]:
y.value_counts()

is_claim
0    54844
1     3748
Name: count, dtype: int64

In [5]:
# Зафиксируем seed для воспроизводимости

def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

In [6]:
# функция перевода класса конфигурации в словарь

def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [ ]:
class CFG:

# Задаем параметры нашего эксперимента

  api = ""# вписать свой API Wandb
  project = "Models"# вписать название эксперимента, который предварительно надо создать в Wandb
  num_epochs = 15 # количество эпох
  train_batch_size = 64 # размер батча обучающей выборки
  test_batch_size = 512 # размер батча тестовой выборки
  num_workers = 2 # количество активных процессов на загрузку данных
  lr = 0.001 # learning_rate
  seed = 2022 # для функции воспроизводимости
  wandb = True # флаг использования Wandb

In [8]:
 #Поделим данные на train, test

X_train, X_test,  y_train, y_test  = train_test_split(X, y, test_size= 0.2, random_state = CFG.seed, stratify = y)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(46873, 89)
(11719, 89)
(46873,)
(11719,)


In [9]:
# Стандартизируем наги значения
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Превращаю в тензор
X_train_tens = torch.tensor(X_train_scaled, dtype  = torch.float32)
X_test_tens = torch.tensor(X_test_scaled, dtype  = torch.float32)
y_train_tens = torch.tensor(y_train.values, dtype  = torch.float32).reshape(-1, 1)
y_test_tens = torch.tensor(y_test.values, dtype  = torch.float32).reshape(-1, 1)


# Создаю train_dataset из X_train_tens и y_train_tens и  test_dataset из X_test_tens и y_test_tens
train_dataset = TensorDataset(X_train_tens, y_train_tens)
test_dataset = TensorDataset(X_test_tens, y_test_tens)

#создаю лоудары, чтобы передавались данные батчами
train_loader = DataLoader(train_dataset, batch_size= CFG.train_batch_size, shuffle= True, num_workers= CFG.num_workers)
test_loader = DataLoader(test_dataset, batch_size= CFG.test_batch_size, shuffle = False, num_workers= CFG.num_workers)




In [10]:
examples = enumerate(train_loader)

batch_ind, (example_data, example_targets ) = next(examples)

In [11]:
example_data.shape

torch.Size([64, 89])

In [12]:
example_targets.shape

torch.Size([64, 1])

## Модель 1

In [13]:
class Model_1 (nn.Module):

    def __init__(self):
        super(Model_1,self).__init__()

        hidden_1 = 256
        hidden_2 = 128
        hidden_3 = 64

        # первый слой (89 -> hidden_1)
        self.fc1 = nn.Linear(89, hidden_1)
        self.batch_norm1 = nn.BatchNorm1d(hidden_1)
        self.act1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        # второй слой (hidden_1 -> hidden_2)
        self.fc2  = nn.Linear(hidden_1, hidden_2)
        self.batch_norm2 = nn.BatchNorm1d(hidden_2)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        # третий слой (hidden_2 -> hidden_3)
        self.fc3  = nn.Linear(hidden_2, hidden_3)
        self.batch_norm3 = nn.BatchNorm1d(hidden_3)
        self.act3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)

        #вывходной слой
        self.fc4 = nn.Linear(hidden_3, 1)

    def forward(self, x):


        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = self.act1(x)
        x = self.dropout1(x)


        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = self.act2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.batch_norm3(x)
        x = self.act3(x)
        x = self.dropout3(x)

        x = self.fc4(x)

        return x

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model_1().to(device)

print(device)
print(model)

cpu
Model_1(
  (fc1): Linear(in_features=89, out_features=256, bias=True)
  (batch_norm1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (batch_norm2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (batch_norm3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=64, out_features=1, bias=True)
)


In [15]:
# функция обучения модели
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score #https://scikit-learn.ru/stable/modules/model_evaluation.html

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()


    train_loss = 0
    correct = 0
    total = 0
    n_ex = len(train_loader)
    all_preds = []
    all_targets = []

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total = n_ex):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        #прямой проход
        output = model(data)
        loss = criterion(output, target)
        train_loss += loss.item()
        probs = torch.sigmoid(output) #считаю вероятность https://docs.pytorch.org/docs/main/generated/torch.nn.Sigmoid.html
        pred = (probs >= 0.55).float() #считаю классы 0/1, если вероятность >= 0.5, ставим класс 1
        correct += pred.eq(target).sum().item()
        total += target.size(0)
        #собраю все предсказания и все реальные ответы со всех батчей в обычные списки, чтобы потом посчитать precision, recall, f1
        all_preds.extend(pred.detach().cpu().numpy().ravel()) #https://docs.pytorch.org/docs/2.12/generated/torch.Tensor.detach.html
        all_targets.extend(target.detach().cpu().numpy().ravel())
        #обратный проход
        loss.backward()
        #градиентный шаг
        optimizer.step()

    #считаю метрики
    train_loss = train_loss / len(train_loader)
    train_accuracy = correct / total
    train_precision = precision_score(all_targets, all_preds, zero_division = 0)
    train_recall = recall_score(all_targets, all_preds, zero_division = 0)
    train_f1 = f1_score(all_targets, all_preds, zero_division = 0)

    tqdm.write('\nTrain Epoch: {} | Average loss: {:.4f} | Accuracy: {:.2f}% | Precision: {:.4f} | Recall: {:.4f} | F1: {:.4f}'.format(epoch,train_loss,100. * train_accuracy,train_precision, train_recall, train_f1))


    if WANDB:
            wandb.log({ 'epoch': epoch, 'train_loss': train_loss,'train_accuracy': train_accuracy,'train_precision': train_precision,'train_recall': train_recall,'train_f1': train_f1})


    return train_loss, train_accuracy, train_precision, train_recall, train_f1


In [16]:
#фунция тестирования
def test(model, device, test_loader, criterion, WANDB=False):
    model.eval()

    test_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            target = target.to(device).float()
            output = model(data)
            loss = criterion(output, target)
            test_loss += loss.item()
            probs = torch.sigmoid(output)
            pred = (probs >= 0.55).float()
            correct += pred.eq(target).sum().item()
            total += target.size(0)
            all_preds.extend(pred.detach().cpu().numpy().ravel())
            all_targets.extend(target.detach().cpu().numpy().ravel())

    test_loss = test_loss / len(test_loader)
    test_accuracy = correct / total

    test_precision = precision_score(all_targets, all_preds, zero_division = 0)
    test_recall = recall_score(all_targets, all_preds, zero_division = 0)
    test_f1 = f1_score(all_targets, all_preds, zero_division = 0)

    tqdm.write('\nTest set: Average loss: {:.4f} | Accuracy: {:.2f}% | Precision: {:.4f} | Recall: {:.4f} | F1: {:.4f}'.format(test_loss,100. * test_accuracy, test_precision, test_recall, test_f1 ) )

    if WANDB:
        wandb.log({'test_loss': test_loss,'test_accuracy': test_accuracy,'test_precision': test_precision, 'test_recall': test_recall, 'test_f1': test_f1})

    return test_loss, test_accuracy, test_precision, test_recall, test_f1





In [17]:
#основная функция для эксперимента
def run_experiment(model, model_name):

    seed_everything(CFG.seed)

    use_cuda= torch.cuda.is_available() #Проверем доступность gpu
    device = torch.device("cuda" if use_cuda else 'cpu') #выделили устройство
    model = model.to(device)



    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype = torch.float32).to(device) #говорим что ошибка на классе 1 важнее, чем ошибка на классе 0
    criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight) #функция потерь
    optimizer = torch.optim.Adam(model.parameters(), lr = CFG.lr) #https://docs.pytorch.org/docs/main/generated/torch.optim.Adam.html




    if CFG.wandb:
        wandb.init(
            project=CFG.project,
            name=model_name,
            config={ **class2dict(CFG), "model_name": model_name, 'architecture' : str(model),
                    'epochs': CFG.num_epochs, 'batch_size': CFG.train_batch_size, 'lr': CFG.lr,
                     'optimizer': 'Adam',  'loss': 'BCEWithLogitsLoss', 'pos_weight': pos_weight.item(),
                     'threshold': 0.55, 'seed': CFG.seed})



    for epoch in range(1, CFG.num_epochs + 1):
        train(model,device,train_loader, optimizer, criterion,epoch, WANDB = CFG.wandb )
        test(model, device, test_loader, criterion, WANDB  = CFG.wandb)
    torch.save(model.state_dict(), f'{model_name}.pth')

    joblib.dump(scaler, 'scaler.pkl') #сохраняем скаллер

    #сохраняем данные
    X_train.to_csv('X_train.csv', index = False)
    X_test.to_csv('X_test.csv', index = False)
    y_train.to_csv('y_train.csv', index = False)
    y_test.to_csv('y_test.csv', index = False)

    if CFG.wandb:

        artifact = wandb.Artifact(name=f'{model_name}_artifacts', type = 'model')

        #добавляю эти файлы в W&B artifact
        artifact.add_file(f'{model_name}.pth')
        artifact.add_file('scaler.pkl')
        artifact.add_file('X_train.csv')
        artifact.add_file('X_test.csv')
        artifact.add_file('y_train.csv')
        artifact.add_file('y_test.csv')
        wandb.log_artifact(artifact)
        wandb.finish()





In [18]:
CFG.wandb

True

In [19]:
run_experiment(model, 'model_1')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/phuongnguyen/.netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


100%|██████████| 733/733 [00:02<00:00, 341.30it/s]



Train Epoch: 1 | Average loss: 1.3023 | Accuracy: 70.27% | Precision: 0.0800 | Recall: 0.3472 | F1: 0.1300

Test set: Average loss: 1.2638 | Accuracy: 77.02% | Precision: 0.1000 | Recall: 0.3240 | F1: 0.1529


100%|██████████| 733/733 [00:02<00:00, 357.35it/s]


Train Epoch: 2 | Average loss: 1.2805 | Accuracy: 71.65% | Precision: 0.0839 | Recall: 0.3459 | F1: 0.1350


...


Test set: Average loss: 1.2404 | Accuracy: 59.62% | Precision: 0.0947 | Recall: 0.6200 | F1: 0.1643


100%|██████████| 733/733 [00:02<00:00, 345.43it/s]


Train Epoch: 15 | Average loss: 1.2492 | Accuracy: 65.42% | Precision: 0.0947 | Recall: 0.5150 | F1: 0.1600



Test set: Average loss: 1.2455 | Accuracy: 73.21% | Precision: 0.1043 | Recall: 0.4200 | F1: 0.1671


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▇▆▄▆▆▆▄▅▅▅▄▄█▁▆
test_f1,▂▂▄▂▁▆▆▇▇▆▄▇▃▇█
test_loss,█▇█▆▆▆▄▃▄▃▄▁▄▁▃
test_precision,▄▄▁▃▂▆▂▆▅▄▁▄█▁▇
test_recall,▂▂▅▂▂▃▅▄▄▄▄▅▁█▄
train_accuracy,▆█▇▇▇▇▇▄▅▃▃▄▂▃▁
train_f1,▁▂▄▅▅▅▅▆▆▆▇▇█▇█
train_loss,█▅▄▄▄▃▃▃▂▂▂▂▁▁▁
train_precision,▁▃▄▆▆▆▆▆▇▆▇▇█▇█
+1,...


Первая модель использовалась как базовая регуляризованная MLP для табличной бинарной классификации. Выбрали архитектуру 89–256–128–64–1, так как она умеренно сложная и достаточно глубокая, чтобы выявлять нелинейные зависимости между признаками и таргетом, но при этом не слишком большая, чтобы резко увеличить риск переобучения. После скрытых слоев применяются BatchNorm1d и Dropout: BatchNorm1d стабилизирует обучение, а Dropout снижает вероятность переобучения. В качестве функции активации используется ReLU, так как она стандартно применяется в полносвязных нейронных сетях и хорошо работает с глубокими архитектурами. Для обучения используется BCEWithLogitsLoss с pos_weight, потому что целевой класс 1 встречается значительно реже класса 0, и без учета дисбаланса модель склонна предсказывать только класс 0

### Воспроизведение без обучения

In [20]:
# Загружаем данные
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

scaler = joblib.load('scaler.pkl') #загружаю скаллер

X_test_scaled = scaler.transform(X_test) #стандартизирую как при обучении




X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32) #перевод в тензор

model = Model_1()

model.load_state_dict(torch.load('model_1.pth', map_location = 'cpu')) #загружаем сохранённые веса обратно в модель

model.eval()

with torch.no_grad(): #Получаем предсказания без обучения
    output = model(X_test_tensor)
    probs = torch.sigmoid(output)
    preds = (probs >= 0.55).float()

print(output.shape)
print(probs[:10])
print(preds[:10])
print(preds.sum())


y_true = y_test.values.ravel()
y_pred = preds.numpy().ravel()

#проверяю качество без обучения
print('accuracy:', accuracy_score(y_true, y_pred))
print('precision:', precision_score(y_true, y_pred, zero_division = 0))
print('recall:', recall_score(y_true, y_pred, zero_division = 0))
print('f1:', f1_score(y_true, y_pred, zero_division = 0))





torch.Size([11719, 1])
tensor([[0.5549],
        [0.4786],
        [0.5798],
        [0.4431],
        [0.4345],
        [0.4719],
        [0.5502],
        [0.5260],
        [0.4873],
        [0.4411]])
tensor([[1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.]])
tensor(3020.)
accuracy: 0.732059049406946
precision: 0.10430463576158941
recall: 0.42
f1: 0.16710875331564987


модель находит много реальных страховых случаев, потому что recall 71.7%, но делает много ложных тревог, потому что precision 9.2%. Поэтому основная метрика f1 = 16.3%

## Модель 2

Вторая модель отличается от первой увеличенной глубиной и шириной сети: вместо архитектуры 89–256–128–64–1 используется 89–512–256–128–64–32–1. Это позволяет проверить, улучшит ли более сложная MLP способность модели выявлять нелинейные зависимости между признаками страхового полиса и фактом наступления страхового случая. Для контроля переобучения сохранены BatchNorm1d и Dropout, причём dropout в первых слоях увеличен

In [21]:
class Model_2 (nn.Module):

    def __init__(self):
        super(Model_2,self).__init__()

        hidden_1 = 512
        hidden_2 = 256
        hidden_3 = 128
        hidden_4 = 64
        hidden_5 = 32

        # первый слой (89 -> hidden_1)
        self.fc1 = nn.Linear(89, hidden_1)
        self.batch_norm1 = nn.BatchNorm1d(512)
        self.act1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)

        #второй слой (hidden_1 -> hidden_2)
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        #третий слой (hidden_2 -> hidden_3)
        self.fc3 = nn.Linear(hidden_2, hidden_3)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.act3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.3)

        #четвертый слой (hidden_3 -> hidden_4)
        self.fc4 = nn.Linear(hidden_3, hidden_4)
        self.batch_norm4 = nn.BatchNorm1d(64)
        self.act4 = nn.ReLU()
        self.dropout4 = nn.Dropout(0.2)

        #пятый слой (hidden_4 -> hidden_5)
        self.fc5 = nn.Linear(hidden_4, hidden_5)
        self.batch_norm5 = nn.BatchNorm1d(32)
        self.act5 = nn.ReLU()
        self.dropout5 = nn.Dropout(0.1)

        #Выходной слой
        self.fc6 = nn.Linear(hidden_5, 1)

    def forward(self, x):

        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = self.act1(x)
        x = self.dropout1(x)


        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = self.act2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.batch_norm3(x)
        x = self.act3(x)
        x = self.dropout3(x)

        x = self.fc4(x)
        x = self.batch_norm4(x)
        x = self.act4(x)
        x = self.dropout4(x)

        x = self.fc5(x)
        x = self.batch_norm5(x)
        x = self.act5(x)
        x = self.dropout5(x)

        x = self.fc6(x)

        return x

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_2 = Model_2().to(device)

print(device)
print(model_2)

cpu
Model_2(
  (fc1): Linear(in_features=89, out_features=512, bias=True)
  (batch_norm1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU()
  (dropout1): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (batch_norm2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (batch_norm3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act3): ReLU()
  (dropout3): Dropout(p=0.3, inplace=False)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (batch_norm4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act4): ReLU()
  (dropout4): Dropout(p=0.2, inplace=False)
  (fc5): Linear(in_features=64, out_features=32, bias=True)
  

In [23]:
run_experiment(model_2, 'model_2')

wandb: Currently logged in as: huesospro2005 (huesospro2005-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 733/733 [00:03<00:00, 217.25it/s]


Train Epoch: 1 | Average loss: 1.2986 | Accuracy: 79.67% | Precision: 0.0761 | Recall: 0.1955 | F1: 0.1095



Test set: Average loss: 1.2666 | Accuracy: 74.15% | Precision: 0.0955 | Recall: 0.3587 | F1: 0.1508


100%|██████████| 733/733 [00:03<00:00, 231.47it/s]


...


Test set: Average loss: 1.2419 | Accuracy: 66.46% | Precision: 0.0954 | Recall: 0.5000 | F1: 0.1602


100%|██████████| 733/733 [00:03<00:00, 233.34it/s]



Train Epoch: 15 | Average loss: 1.2515 | Accuracy: 65.52% | Precision: 0.0925 | Recall: 0.4980 | F1: 0.1560

Test set: Average loss: 1.2493 | Accuracy: 65.34% | Precision: 0.0929 | Recall: 0.5040 | F1: 0.1569


wandb: WARNING Artifact "model_2_artifacts" already exists with the same content. No new version will be created.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▆▃█▆▃▆█▃▅▃▁▄▃▄▄
test_f1,▄▄▁▂▄▄▃▆▆▇▆▆█▇▆
test_loss,▆█▅▄▅▄▃▃▂▂▂▂▁▁▂
test_precision,▃▁▆▂▁▄█▃▄▃▂▃▄▃▃
test_recall,▃▆▁▃▆▃▁▆▄▆█▅▇▆▆
train_accuracy,█▆▄▄▄▃▃▂▃▂▂▁▁▁▁
train_f1,▁▄▆▆▆▇▇▆█▇█████
train_loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
train_precision,▁▅▆▆▆▇▇▆█▇▇█▇▇█
+1,...


Вторая модель была построена как более глубокая и широкая версия базовой MLP. По сравнению с первой конфигурацией архитектура была увеличена с 89–256–128–64–1 до 89–512–256–128–64–32–1. Основная цель это проверить, сможет ли более сложная нейронная сеть лучше выявлять нелинейные зависимости между характеристиками автомобиля, страхового полиса и фактом наступления страхового случая. Функция активации ReLU, оптимизатор Adam, learning rate, batch size и количество эпох были оставлены такими же, как в первой модели, так как они были оптимальными для этих данных + сравним только влияние архитектурных изменений. Для контроля переобучения использовались BatchNorm1d и Dropout. 

модель 2 лучше ловит страховые случаи, чем модель 1: recall вырос с 0.42 до 0.504. Но precision и F1 ниже, поэтому она делает больше ложных срабатываний и пока выглядит хуже модели 1 по общему балансу

## Воспроизведение без обучения

In [24]:
# Загружаем данные
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

scaler = joblib.load('scaler.pkl') #загружаю скаллер

X_test_scaled = scaler.transform(X_test) #стандартизирую как при обучении




X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32) #перевод в тензор

model = Model_2()

model.load_state_dict(torch.load('model_2.pth', map_location = 'cpu')) #загружаем сохранённые веса обратно в модель

model.eval()

with torch.no_grad(): #Получаем предсказания без обучения
    output = model(X_test_tensor)
    probs = torch.sigmoid(output)
    preds = (probs >= 0.55).float()

print(output.shape)
print(probs[:10])
print(preds[:10])
print(preds.sum())


y_true = y_test.values.ravel()
y_pred = preds.numpy().ravel()

#проверяю качество без обучения
print('accuracy:', accuracy_score(y_true, y_pred))
print('precision:', precision_score(y_true, y_pred, zero_division = 0))
print('recall:', recall_score(y_true, y_pred, zero_division = 0))
print('f1:', f1_score(y_true, y_pred, zero_division = 0))

torch.Size([11719, 1])
tensor([[0.5582],
        [0.5168],
        [0.5896],
        [0.4202],
        [0.4016],
        [0.4898],
        [0.5814],
        [0.4266],
        [0.5340],
        [0.4982]])
tensor([[1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.]])
tensor(4068.)
accuracy: 0.6533833944875843
precision: 0.09292035398230089
recall: 0.504
f1: 0.1569115815691158


## Модель 3

Третья модель - Wide & Deep MLP. У нее две ветки: deep ветка пропускает признаки через несколько Linear слоев и ищет сложные зависимости, а wide ветка почти напрямую передает исходные признаки дальше. Потом две ветки объединяются через concatenation, и после этого идет финальный классификатор. Wide часть сохраняет простые связи, Dеер часть ищет сложные связи. К тому же мы выбрали AdamW в качестве оптимизатора вместо Adam, потому что в третьей модели используется более сложная архитектура и weight_decay, а AdamW корректнее применяет L2-регуляризацию, поэтому лучше подходит для контроля переобучения

In [25]:
class Model_3(nn.Module):
    def __init__(self):
        super(Model_3, self).__init__()

        hidden_1_deep = 256
        hidden_2_deep = 128
        hidden_3_deep = 64
        hidden_1_wide = 64

        # deep-ветка: ищет сложные зависимости
        self.deep_fc1 = nn.Linear(89, hidden_1_deep)
        self.deep_bn1 = nn.BatchNorm1d(256)
        self.deep_act1 = nn.ReLU()
        self.deep_dropout1 = nn.Dropout(0.3)

        self.deep_fc2 = nn.Linear(hidden_1_deep, hidden_2_deep)
        self.deep_bn2 = nn.BatchNorm1d(128)
        self.deep_act2 = nn.ReLU()
        self.deep_dropout2 = nn.Dropout(0.2)

        self.deep_fc3 = nn.Linear(hidden_2_deep, hidden_3_deep)
        self.deep_bn3 = nn.BatchNorm1d(64)
        self.deep_act3 = nn.ReLU()
        self.deep_dropout3 = nn.Dropout(0.2)

        # wide-ветка: сохраняет простые зависимости
        self.wide_fc = nn.Linear(89, hidden_1_wide)
        self.wide_bn = nn.BatchNorm1d(64)
        self.wide_act = nn.ReLU()

        # после объединения deep 64 + wide 64 = 128
        self.final_fc1 = nn.Linear(128, 64)
        self.final_bn1 = nn.BatchNorm1d(64)
        self.final_act1 = nn.ReLU()
        self.final_dropout1 = nn.Dropout(0.2)

        self.final_fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # deep-ветка
        deep = self.deep_fc1(x)
        deep = self.deep_bn1(deep)
        deep = self.deep_act1(deep)
        deep = self.deep_dropout1(deep)

        deep = self.deep_fc2(deep)
        deep = self.deep_bn2(deep)
        deep = self.deep_act2(deep)
        deep = self.deep_dropout2(deep)

        deep = self.deep_fc3(deep)
        deep = self.deep_bn3(deep)
        deep = self.deep_act3(deep)
        deep = self.deep_dropout3(deep)

        # wide-ветка
        wide = self.wide_fc(x)
        wide = self.wide_bn(wide)
        wide = self.wide_act(wide)

        # объединяем две ветки
        x = torch.cat([deep, wide], dim=1)


        # финальный классификатор
        x = self.final_fc1(x)
        x = self.final_bn1(x)
        x = self.final_act1(x)
        x = self.final_dropout1(x)

        x = self.final_fc2(x)

        return x

In [26]:
# перепишем основную функцию для эксперимента, добавив weight_decay и поменяв оптимизатор
def run_experiment_2(model, model_name):

    seed_everything(CFG.seed)

    use_cuda= torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else 'cpu')
    model = model.to(device)


    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype = torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001, weight_decay = 0.0001) #добавляем L2-регуляризация https://docs.pytorch.org/docs/main/generated/torch.optim.AdamW.html

    num_epochs = 15
    if CFG.wandb:
        wandb.init(
            project=CFG.project,
            name=model_name,
            config={ **class2dict(CFG), "model_name": model_name, 'architecture' : str(model),
                    'epochs': num_epochs, 'batch_size': CFG.train_batch_size, 'lr': 0.0005, 'optimizer': 'AdamW',
                     'loss': 'BCEWithLogitsLoss', 'pos_weight': pos_weight.item(), 'weight_decay': 0.0001,
                     'threshold': 0.55, 'seed': CFG.seed})


    for epoch in range(1, num_epochs + 1):
        train(model,device,train_loader, optimizer, criterion,epoch, WANDB = CFG.wandb )
        test(model, device, test_loader, criterion,WANDB  = CFG.wandb)
    torch.save(model.state_dict(), f'{model_name}.pth')

    joblib.dump(scaler, 'scaler.pkl') #сохраняем скаллер

    #сохраняем данные
    X_train.to_csv('X_train.csv', index = False)
    X_test.to_csv('X_test.csv', index = False)
    y_train.to_csv('y_train.csv', index = False)
    y_test.to_csv('y_test.csv', index = False)

    if CFG.wandb:

        artifact = wandb.Artifact(name=f'{model_name}_artifacts', type = 'model')

        #добавляю эти файлы в W&B artifact
        artifact.add_file(f'{model_name}.pth')
        artifact.add_file('scaler.pkl')
        artifact.add_file('X_train.csv')
        artifact.add_file('X_test.csv')
        artifact.add_file('y_train.csv')
        artifact.add_file('y_test.csv')
        wandb.log_artifact(artifact)
        wandb.finish()


In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_3 = Model_3().to(device)

print(device)
print(model_3)

cpu
Model_3(
  (deep_fc1): Linear(in_features=89, out_features=256, bias=True)
  (deep_bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (deep_act1): ReLU()
  (deep_dropout1): Dropout(p=0.3, inplace=False)
  (deep_fc2): Linear(in_features=256, out_features=128, bias=True)
  (deep_bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (deep_act2): ReLU()
  (deep_dropout2): Dropout(p=0.2, inplace=False)
  (deep_fc3): Linear(in_features=128, out_features=64, bias=True)
  (deep_bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (deep_act3): ReLU()
  (deep_dropout3): Dropout(p=0.2, inplace=False)
  (wide_fc): Linear(in_features=89, out_features=64, bias=True)
  (wide_bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (wide_act): ReLU()
  (final_fc1): Linear(in_features=128, out_features=64, bias=True)
  

In [28]:
run_experiment_2(model_3, 'model_3')

100%|██████████| 733/733 [00:02<00:00, 254.88it/s]


Train Epoch: 1 | Average loss: 1.2970 | Accuracy: 71.99% | Precision: 0.0802 | Recall: 0.3229 | F1: 0.1285



Test set: Average loss: 1.2543 | Accuracy: 69.64% | Precision: 0.0945 | Recall: 0.4360 | F1: 0.1553


100%|██████████| 733/733 [00:02<00:00, 258.77it/s]


Train Epoch: 2 | Average loss: 1.2735 | Accuracy: 68.46% | Precision: 0.0860 | Recall: 0.4086 | F1: 0.1421


...

100%|██████████| 733/733 [00:02<00:00, 251.17it/s]


Train Epoch: 15 | Average loss: 1.2382 | Accuracy: 67.51% | Precision: 0.0973 | Recall: 0.4927 | F1: 0.1624



Test set: Average loss: 1.2408 | Accuracy: 62.07% | Precision: 0.0986 | Recall: 0.6053 | F1: 0.1696


wandb: WARNING Artifact "model_3_artifacts" already exists with the same content. No new version will be created.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▆▄▇▃▅█▂▁▅▅▅▅▂▆▂
test_f1,▁▁▅▃▆▇▅▄▅▅▁▅▅▇█
test_loss,▆█▇▄▄▇▄▅▃▃▆▁▁▂▁
test_precision,▂▁▆▂▅█▂▂▄▄▂▄▂▆▄
test_recall,▂▅▁▅▄▁▇█▄▄▃▄▇▃█
train_accuracy,█▄▅▇▄▅▅▄▂▃▃▂▂▁▃
train_f1,▁▄▅▅▆▆▆▆▆▇▇████
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▁▁▁
train_precision,▁▃▅▆▆▆▆▆▆▇▇▇▇██
+1,...


При первом запуске третья модель Wide & Deep MLP показала наилучшее значение F1-меры среди всех рассмотренных конфигураций: 0.169. Также третья модель достигла самого высокого recall — 0.636, что означает лучшую способность выявлять объекты положительного класса, то есть случаи страхового требования. Это показывает, что наша новая архитектура с ветками эффективнее для данной задачи, чем обычная последовательная MLP. Попробуем еще оптимизировать ее.

In [29]:
# Загружаем данные
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

scaler = joblib.load('scaler.pkl') #загружаю скаллер

X_test_scaled = scaler.transform(X_test) #стандартизирую как при обучении




X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32) #перевод в тензор

model = Model_3()

model.load_state_dict(torch.load('model_3.pth', map_location = 'cpu')) #загружаем сохранённые веса обратно в модель

model.eval()

with torch.no_grad(): #Получаем предсказания без обучения
    output = model(X_test_tensor)
    probs = torch.sigmoid(output)
    preds = (probs >= 0.55).float()

print(output.shape)
print(probs[:10])
print(preds[:10])
print(preds.sum())


y_true = y_test.values.ravel()
y_pred = preds.numpy().ravel()

#проверяю качество без обучения
print('accuracy:', accuracy_score(y_true, y_pred))
print('precision:', precision_score(y_true, y_pred, zero_division = 0))
print('recall:', recall_score(y_true, y_pred, zero_division = 0))
print('f1:', f1_score(y_true, y_pred, zero_division = 0))

torch.Size([11719, 1])
tensor([[0.5618],
        [0.5115],
        [0.6293],
        [0.4655],
        [0.4749],
        [0.5247],
        [0.5796],
        [0.6094],
        [0.6155],
        [0.4960]])
tensor([[1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.]])
tensor(4603.)
accuracy: 0.6207014250362659
precision: 0.09863132739517706
recall: 0.6053333333333333
f1: 0.1696245096207734


В итоге лучшей моделью стала модель 3. Но, как можно заметить, значения нашей главной метрики f1 там тоже не сильно велико. Наше предположение - это потому, что по собранным признакам тяжело предсказать то, обратится клиент за выплатой или нет. И действительно, после проверки (файл с EDA) оказалось, что зависимость у признаков довольна низкая с таргетом и добавление еще новых признаков с помошью feature engeeniring не дало признаков с корреляцией по модулю > 0.1. А это очень низкие значения корреляции.
Возвращаясь к нашей исходной бизнес задаче: предсказания того, обратится ли клиент за страховой выплатой является довольно сложной задачей в силу того, что чисто логически тяжело выявить сильную зависимость между какими-то характеристиками и обращенияем, потому что часто все же различные инциденты являются последствиями неудачного стечения обстоятельств, а так же в большой степени зависят не только от поведения водителя и проч, но и поведения других участников дорожного движения.
Таким образом, мы бы предложили пользоваться нашей моделью для первичного анализа и использовать ее в качестве дополнительного инструмента оценки риска. Например, страховая компания может использовать прогноз модели для разделения клиентов на группы риска. Клиенты с прогнозом 1 могут быть направлены на более детальную проверку, дополнительный анализ или включены в программы управления рисками.
Также модель может применяться для приоритизации работы аналитиков и специалистов по страхованию. Вместо проверки всех клиентов одинаково компания сможет уделять больше внимания тем случаям, которые модель считает наиболее вероятными для последующего обращения за выплатой.
По мере накопления новых данных модель может быть дообучена и улучшена, что позволит постепенно повышать качество прогнозирования и расширять область ее применения.